# Notebook 1 — Bigram Language Model on `names.txt`

이 노트북의 목표는 **가장 기본적인 문자 단위 language model**을 만드는 것입니다.

- 입력: 현재 문자 1개
- 출력: 다음 문자 1개
- 즉, **bigram** 확률표를 학습합니다.

전체 구조는 `Dataset → DataLoader → Model → Loss → Train loop → Sampling`입니다.

## 1. 데이터 준비

In [2]:
# 파이토치(PyTorch) 라이브러리를 불러옵니다. 데이터 분석의 Numpy나 Pandas 같은 역할입니다.
import torch
# 신경망(Neural Network) 모델을 구성하는 기본 뼈대들이 들어있는 모듈입니다.
import torch.nn as nn
# 신경망에서 자주 쓰이는 함수들(활성화 함수, 손실 함수 등)이 들어있습니다.
import torch.nn.functional as F
# 데이터를 모델에 조금씩 나누어 먹여주는(Batch) 역할을 하는 도구들입니다.
from torch.utils.data import Dataset, DataLoader
# 파일 경로를 쉽게 다루기 위한 파이썬 기본 라이브러리입니다.
from pathlib import Path

# "names.txt" 파일이 현재 폴더에 없다면, 깃허브에서 다운로드 받습니다 (!wget은 리눅스 명령어)
if not Path("names.txt").exists():
    !wget -q https://raw.githubusercontent.com/karpathy/makemore/master/names.txt

# 파일을 읽기 모드("r")로 열고, 전체 텍스트를 읽은 뒤, 줄바꿈(엔터)을 기준으로 잘라 리스트로 만듭니다.
# 결과적으로 words = ['emma', 'olivia', 'ava', ...] 형태가 됩니다.
words = open("names.txt", "r").read().splitlines()

# 1. "".join(words): 모든 이름을 하나의 거대한 문자열로 합칩니다. ("emmaoliviaava...")
# 2. set(...): 중복을 제거하여 텍스트에 사용된 고유한 알파벳만 남깁니다.
# 3. list(...) -> sorted(...): 고유한 알파벳들을 리스트로 만들고 알파벳 순서대로 정렬합니다.
chars = sorted(list(set("".join(words))))

# '.'을 리스트의 맨 앞에 추가합니다.
# 이 마침표는 단어의 시작과 끝을 알려주는 '특수 기호' 역할을 할 것입니다.
chars = ['.'] + chars

# 문자를 주면 숫자(인덱스)를 반환하는 사전(Dictionary)을 만듭니다. (예: '.' -> 0, 'a' -> 1)
stoi = {ch: i for i, ch in enumerate(chars)}
# 반대로 숫자(인덱스)를 주면 문자를 반환하는 사전을 만듭니다. 나중에 결과를 사람의 언어로 읽기 위해 필요합니다.
itos = {i: ch for ch, i in stoi.items()}

# 우리가 다룰 전체 고유 문자(알파벳+마침표)의 개수입니다. (총 27개)
vocab_size = len(stoi)

# 모든 이름(단어)을 숫자의 나열로 변환합니다.
# 예: 'emma' -> [5, 13, 13, 1] (사전에 정의된 숫자로 치환됨)
encoded_words = [[stoi[ch] for ch in w] for w in words]

print("num words:", len(words)) # 총 단어(이름) 개수 출력
print("vocab_size:", vocab_size) # 고유 문자 개수 출력 (27)
print(words[:10]) # 첫 10개 이름 확인

num words: 32033
vocab_size: 27
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia', 'harper', 'evelyn']


## 2. Dataset (`block_size = 1`)

In [3]:
# 파이토치의 Dataset 클래스를 상속받아 나만의 데이터셋 규칙을 정의합니다.
class NamesContextDataset(Dataset):
    def __init__(self, encoded_words, block_size):
        # X는 입력 데이터(문제), Y는 출력 데이터(정답)를 담을 빈 리스트입니다.
        self.X, self.Y = [], []

        for word in encoded_words:
            # context는 모델이 참고할 이전 글자들의 묶음입니다.
            # 처음에는 0 (마침표 '.')으로 채워 넣습니다. "단어의 시작"을 의미합니다.
            context = [0] * block_size

            # 단어 끝에 0 (마침표 '.')을 덧붙여서 "단어의 끝"을 알려줍니다.
            for ix in word + [0]:
                self.X.append(context.copy()) # 현재 상태(문제)를 X에 저장
                self.Y.append(ix)             # 다음에 올 글자(정답)를 Y에 저장

                # context를 한 칸 옆으로 밉니다.
                # (예: [0] 이었다면, 다음번엔 입력된 글자 ix가 새로운 context가 됩니다.)
                context = context[1:] + [ix]

        # 리스트로 모은 X와 Y를 파이토치의 '텐서(Tensor)'로 변환합니다.
        # 텐서는 GPU 연산에 최적화된 고성능 다차원 배열(Numpy array와 유사)입니다.
        # dtype=torch.long은 정수형 데이터를 의미합니다.
        self.X = torch.tensor(self.X, dtype=torch.long)
        self.Y = torch.tensor(self.Y, dtype=torch.long)

    # 데이터의 총 개수를 반환하는 함수 (파이토치가 내부적으로 사용함)
    def __len__(self):
        return len(self.Y)

    # 특정 인덱스(idx)의 문제(X)와 정답(Y)을 하나씩 꺼내주는 함수
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

# block_size가 1이라는 것은 오직 "바로 직전 글자 1개"만 보고 다음 글자를 예측한다는 뜻입니다. (이것이 Bigram 모델)
block_size = 1
dataset = NamesContextDataset(encoded_words, block_size)

# DataLoader는 수만 개의 데이터를 한 번에 다 학습시키면 메모리가 터지므로,
# batch_size(여기선 32)만큼 데이터를 묶어서 무작위로(shuffle=True) 던져주는 유용한 도구입니다.
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# 데이터가 잘 만들어졌는지 1번째 데이터를 하나만 뽑아봅니다.
x, y = dataset[1]
# DataLoader에서 32개 묶음(배치)을 하나 뽑아봅니다.
xb, yb = next(iter(loader))

print("single x:", x, x.shape)
print("single y:", y)
print("batch xb.shape:", xb.shape) # 32개의 문제(X) 묶음 (크기: 32x1)
print("batch yb.shape:", yb.shape) # 32개의 정답(Y) 묶음 (크기: 32)

single x: tensor([5]) torch.Size([1])
single y: tensor(13)
batch xb.shape: torch.Size([32, 1])
batch yb.shape: torch.Size([32])


## 3. Bigram model (명시적 one-hot 버전)

In [4]:
# 파이토치의 nn.Module을 상속받아 모델을 만듭니다. (모든 딥러닝 모델의 기본 규칙)
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.vocab_size = vocab_size

        # 가중치 행렬 W를 만듭니다. (단어장 크기 x 단어장 크기 = 27 x 27)
        # torch.randn: 평균 0, 표준편차 1인 정규분포에서 무작위 숫자를 뽑아 초기 가중치를 설정합니다.
        # nn.Parameter: "이 행렬은 고정된 숫자가 아니라, 학습을 통해 정답에 가깝게 업데이트할 변수야!"라고 파이토치에 알려주는 역할입니다.
        self.W = nn.Parameter(torch.randn(vocab_size, vocab_size))

    # forward는 데이터가 모델에 들어와서 예측값을 계산하는 과정(정방향 연산)을 정의합니다.
    def forward(self, x):
        # 2차원(32x1) 데이터를 1차원(32)으로 평평하게 폅니다.
        x = x.view(-1)

        # One-hot 인코딩: 숫자 인덱스를 벡터로 바꿉니다.
        # 예: 3 -> [0, 0, 0, 1, 0, 0, ... 0] (해당 숫자의 위치만 1, 나머지는 0)
        # float(): 연산을 위해 정수를 실수형으로 바꿉니다.
        x_onehot = F.one_hot(x, num_classes=self.vocab_size).float()

        # @ 기호는 행렬 곱셈(Matrix Multiplication)을 의미합니다.
        # 입력된 글자(x_onehot)와 가중치(W)를 곱해서 다음 글자에 대한 점수(logits)를 계산합니다.
        # logits은 확률로 변환되기 전의 '날 것의 예측 점수'를 뜻합니다.
        logits = x_onehot @ self.W
        return logits

model = BigramLanguageModel(vocab_size)
# 32개 묶음 데이터를 모델에 넣어 점수(logits)를 얻어봅니다.
logits = model(xb)
print("logits.shape:", logits.shape) # 결과 크기: (32, 27) -> 32개의 문제 각각에 대해 27개 알파벳의 확률 점수가 나옴

# F.cross_entropy는 모델의 예측 점수(logits)와 실제 정답(yb)을 비교해서
# "얼마나 틀렸는지"를 수치로 나타내는 '손실 함수(Loss Function)'입니다. 값이 낮을수록 똑똑한 모델입니다.
print("initial loss:", F.cross_entropy(logits, yb).item())

logits.shape: torch.Size([32, 27])
initial loss: 3.6353678703308105


In [5]:
# @title
import torch
import torch.nn as nn
import torch.nn.functional as F

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, x):
        x = x.view(-1)  # (B,)
        logits = self.token_embedding_table(x)  # (B, V)
        return logits

## 4. Train / Eval 함수

In [6]:
def train_one_epoch(model, loader, optimizer, device):
    # 모델을 '학습 모드'로 전환합니다. (일부 딥러닝 기법들은 학습/평가 때 다르게 동작하기 때문에 습관적으로 씁니다)
    model.train()
    total_loss = 0.0

    # DataLoader가 32개씩 묶어주는 데이터를 끝까지 순회합니다. (이 한 바퀴를 1 Epoch이라고 부릅니다)
    for xb, yb in loader:
        # 데이터를 CPU나 GPU(device) 등 연산할 곳으로 보냅니다.
        xb, yb = xb.to(device), yb.to(device)

        # 1. 예측 (Forward): 모델에 문제를 넣고 예측 점수를 받습니다.
        logits = model(xb)
        # 2. 오차 계산: 예측 점수와 정답을 비교해 오차(loss)를 계산합니다.
        loss = F.cross_entropy(logits, yb)

        # 3. 기울기 초기화: 파이토치는 오차의 기울기(Gradient)를 계속 누적하는 특징이 있어서,
        #    새로운 계산을 위해 이전 계산의 흔적을 0으로 싹 비워줘야 합니다.
        optimizer.zero_grad()
        # 4. 역전파 (Backward): "오차를 줄이려면 W 가중치들을 각각 어느 방향으로 얼마나 수정해야 할까?"
        #    그 미분값(기울기)을 자동으로 쫘악 계산해줍니다. (딥러닝의 핵심!)
        loss.backward()
        # 5. 가중치 업데이트 (Step): 위에서 구한 미분값을 바탕으로 가중치 W를 실제로 살짝 수정합니다.
        optimizer.step()

        # 손실값(loss)을 전체 데이터 개수만큼 곱해서 누적해 둡니다. 나중에 평균을 내기 위함입니다.
        total_loss += loss.item() * xb.size(0)

    # 전체 오차의 평균을 반환합니다.
    return total_loss / len(loader.dataset)

# @torch.no_grad(): "지금부터는 학습(가중치 업데이트)을 안 할 거니까,
# 복잡한 미분 계산을 위한 메모리를 쓰지 마!" 라고 선언하는 데코레이터입니다. 속도와 메모리를 아껴줍니다.
@torch.no_grad()
def evaluate(model, loader, device):
    # 모델을 '평가(실전) 모드'로 전환합니다.
    model.eval()
    total_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = F.cross_entropy(logits, yb) # 정답과 얼마나 차이나는지 점수만 확인합니다. (수정 작업 없음)
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)

## 5. 학습

In [7]:
# 그래픽카드(GPU)가 있으면 "cuda"를 써서 연산을 엄청나게 빠르게 하고, 없으면 기본 CPU를 사용합니다.
device = "cuda" if torch.cuda.is_available() else "cpu"

# 모델을 생성하고, 방금 정한 장치(CPU or GPU)로 올려보냅니다.
model = BigramLanguageModel(vocab_size).to(device)

# 최적화 알고리즘(Optimizer)을 선택합니다.
# AdamW는 경사하강법(Gradient Descent)의 발전된 형태 중 하나로,
# 가중치를 어느 정도의 보폭(lr: learning rate, 학습률)으로 수정할지 스마트하게 조절해주는 유명한 도구입니다.
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-1)

# 총 20번(20 epoch) 데이터셋 전체를 훑으며 반복 학습합니다.
for epoch in range(20):
    # 학습을 진행하고 오차 평균값을 돌려받습니다.
    train_loss = train_one_epoch(model, loader, optimizer, device)

    # 매 5번째 바퀴, 그리고 마지막(19) 바퀴마다 오차가 얼마나 줄어들었는지 화면에 출력합니다.
    # 점점 수치가 낮아진다면 모델이 패턴을 성공적으로 찾고 있다는 뜻입니다.
    if epoch % 5 == 0 or epoch == 19:
        print(f"epoch {epoch:2d} | train loss {train_loss:.4f}")

epoch  0 | train loss 2.5283
epoch  5 | train loss 2.5239
epoch 10 | train loss 2.5233
epoch 15 | train loss 2.5239
epoch 19 | train loss 2.5239


## 6. Sampling

In [9]:
@torch.no_grad()
def sample(model, block_size, itos, device, num_samples=10, max_len=20):
    model.eval() # 평가(생성) 모드
    results = [] # 만들어진 이름들을 담을 리스트

    # 10개(num_samples)의 이름을 생성하기 위해 10번 반복합니다.
    for _ in range(num_samples):
        # 시작은 항상 0(마침표)입니다. "단어를 시작해라!"라는 신호를 모델에 줍니다.
        context = torch.zeros((1, block_size), dtype=torch.long, device=device)
        out = [] # 뱉어낸 글자들을 차곡차곡 담을 리스트

        # 최대 20글자(max_len)까지 생성합니다.
        for _ in range(max_len):
            # 1. 모델에 현재 글자(context)를 넣고 다음 27개 알파벳에 대한 예측 점수(logits)를 뽑아냅니다.
            logits = model(context)

            # 2. F.softmax: -1.5, 3.2 같은 '날 것의 점수(logits)'를 다 합치면 1.0(100%)이 되도록
            #    0~1 사이의 확률값으로 예쁘게 바꿔주는 함수입니다.
            probs = F.softmax(logits, dim=-1)

            # 3. torch.multinomial: 확률을 기반으로 주사위를 굴립니다.
            #    단순히 확률이 제일 높은 것만 뽑는 게 아니라, 60% 확률인 글자는 60% 빈도로, 10% 확률인 글자는 10% 빈도로
            #    랜덤성을 부여해서 뽑아줍니다. (그래서 실행할 때마다 생성되는 이름이 다릅니다)
            ix = torch.multinomial(probs, num_samples=1)
            next_token = ix.item()

            # 만약 뽑힌 글자가 0(마침표, 단어의 끝)이라면 생성을 즉시 멈춥니다(break).
            if next_token == 0:
                break

            # 4. 뽑힌 숫자(인덱스)를 itos 사전을 통해 다시 사람이 읽을 수 있는 문자로 바꾸어 out에 기록합니다.
            out.append(itos[next_token])

            # 5. 방금 뽑힌 글자를 다음 예측의 입력(context)으로 사용하기 위해 context를 갱신합니다.
            context = torch.cat([context[:, 1:], ix], dim=1)

        # 생성된 알파벳들을 하나로 붙여서("".join) 리스트에 추가합니다.
        results.append("".join(out))

    return results

# 학습된 모델을 통해 10개의 가짜 이름을 생성해봅니다.
sample(model, block_size, itos, device, num_samples=10)

['by',
 'glaha',
 'weleczaeleroaina',
 'shice',
 'janlevidistalialvlaw',
 'ja',
 'loniny',
 'aarayleniazialihmema',
 'marirethvantiny',
 'fia']

## 7. 정리

- `block_size=1`이면 bigram 문제를 정확히 표현할 수 있습니다.
- one-hot과 행렬곱만으로도 language model을 만들 수 있습니다.
- 이 학습 루프는 이후 단계에서도 거의 그대로 재사용됩니다.